## **Fine-tuning BERT for named-entity recognition**

In this notebook, we are going to use **BertForTokenClassification** which is included in the [Transformers library](https://github.com/huggingface/transformers) by HuggingFace. This model has BERT as its base architecture, with a token classification head on top, allowing it to make predictions at the token level, rather than the sequence level. Named entity recognition is typically treated as a token classification problem, so that's what we are going to use it for.

This tutorial uses the idea of **transfer learning**, i.e. first pretraining a large neural network in an unsupervised way, and then fine-tuning that neural network on a task of interest. In this case, BERT is a neural network pretrained on 2 tasks: masked language modeling and next sentence prediction. Now, we are going to fine-tune this network on a NER dataset. Fine-tuning is supervised learning, so this means we will need a labeled dataset.

If you want to know more about BERT, I suggest the following resources:
* the original [paper](https://arxiv.org/abs/1810.04805)
* Jay Allamar's [blog post](http://jalammar.github.io/illustrated-bert/) as well as his [tutorial](http://jalammar.github.io/a-visual-guide-to-using-bert-for-the-first-time/)
* Chris Mccormick's [Youtube channel](https://www.youtube.com/channel/UCoRX98PLOsaN8PtekB9kWrw)
* Abbishek Kumar Mishra's [Youtube channel](https://www.youtube.com/user/abhisheksvnit)

The following notebook largely follows the same structure as the tutorials by Abhishek Kumar Mishra. For his tutorials on the Transformers library, see his [Github repository](https://github.com/abhimishra91/transformers-tutorials).

NOTE: this notebook assumes basic knowledge about deep learning, BERT, and native PyTorch. If you want to learn more Python, deep learning and PyTorch, I highly recommend cs231n by Stanford University and the FastAI course by Jeremy Howard et al. Both are freely available on the web.  

Now, let's move on to the real stuff!

#### **Importing Python Libraries and preparing the environment**

This notebook assumes that you have the following libraries installed:
* pandas
* numpy
* sklearn
* pytorch
* transformers
* seqeval
* datasets

We'll install the required packages using `uv`. If running on Google Colab, you can use `pip install` instead.

In [1]:
!pip install transformers seqeval pandas numpy scikit-learn torch --quiet
!pip install datasets huggingface_hub tokenizers transformers --quiet

In [2]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, BertConfig, BertForTokenClassification
from datasets import load_dataset

As deep learning can be accellerated a lot using a GPU instead of a CPU, make sure you can run this notebook in a GPU runtime (which Google Colab provides for free! - check "Runtime" - "Change runtime type" - and set the hardware accelerator to "GPU").

We can set the default device to GPU using the following code (if it prints "cuda", it means the GPU has been recognized):

In [4]:
from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'
print(device)

cuda


#### **Downloading and preprocessing the data**
We use the CoNLL-2003 dataset, where each word in a sentence is labeled with its named-entity tag using the BIO scheme.

In [5]:
from datasets import load_dataset, Dataset

# Load CoNLL-2003 via the parquet export (avoids the loading-script decode error).
raw = load_dataset("tner/conll2003", revision="refs/convert/parquet")

# tner label ordering
tner_id2label = {0:'O', 1:'B-ORG', 2:'B-MISC', 3:'B-PER', 4:'I-PER',
                 5:'B-LOC', 6:'I-ORG', 7:'I-MISC', 8:'I-LOC'}

# standard ordering for the rest of the notebook
label_names = ['O','B-PER','I-PER','B-ORG','I-ORG','B-LOC','I-LOC','B-MISC','I-MISC']
std = {name: i for i, name in enumerate(label_names)}

def fix(split):
    new_tokens, new_tags = [], []
    for words, ids in zip(split['tokens'], split['tags']):
        new_tokens.append(list(words))
        new_tags.append([std[tner_id2label[i]] for i in ids])
    return Dataset.from_dict({'tokens': new_tokens, 'ner_tags': new_tags})

dataset = {
    'train':      fix(raw['train']),
    'validation': fix(raw['validation']),
    'test':       fix(raw['test']),
}

print("Loaded CoNLL-2003 (real data).")
for k, v in dataset.items():
    print(f"  {k}: {len(v)} examples")
print("\nExample:")
print(dataset['train'][0])

Loaded CoNLL-2003 (real data).
  train: 14041 examples
  validation: 3250 examples
  test: 3453 examples

Example:
{'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}


Let's check the NER tags available in this dataset:

In [6]:
# Get the label names from the dataset
print(f"Number of tags: {len(label_names)}")
print(f"Tags: {label_names}")

Number of tags: 9
Tags: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


The CoNLL-2003 dataset has 9 NER tags:
- O: Outside any named entity
- B-PER, I-PER: Beginning/Inside of a person name
- B-ORG, I-ORG: Beginning/Inside of an organization
- B-LOC, I-LOC: Beginning/Inside of a location
- B-MISC, I-MISC: Beginning/Inside of a miscellaneous entity (e.g. nationalities, events)

Let's check the dataset sizes:

In [7]:
print(f"Train: {len(dataset['train'])} examples")
print(f"Validation: {len(dataset['validation'])} examples")
print(f"Test: {len(dataset['test'])} examples")

Train: 14041 examples
Validation: 3250 examples
Test: 3453 examples


Let's convert the dataset to the format we need for training. We'll create sentence strings and corresponding label strings:

In [8]:
def convert_to_dataframe(hf_dataset, label_names):
    """Convert HuggingFace dataset to pandas DataFrame with sentence and word_labels columns."""
    sentences = []
    word_labels = []

    for example in hf_dataset:
        sentence = " ".join(example['tokens'])
        sentences.append(sentence)
        labels = ",".join([label_names[tag_id] for tag_id in example['ner_tags']])
        word_labels.append(labels)

    return pd.DataFrame({'sentence': sentences, 'word_labels': word_labels})

# Convert train and test splits
train_data = convert_to_dataframe(dataset['train'], label_names)
test_data = convert_to_dataframe(dataset['test'], label_names)

print(f"Train DataFrame shape: {train_data.shape}")
print(f"Test DataFrame shape: {test_data.shape}")
train_data.head()

Train DataFrame shape: (14041, 2)
Test DataFrame shape: (3453, 2)


,sentence,word_labels
0,EU rejects German call to boycott British lamb .,"B-ORG,O,B-MISC,O,O,O,B-MISC,O,O"
1,Peter Blackburn,"B-PER,I-PER"
2,BRUSSELS 1996-08-22,"B-LOC,O"
3,The European Commission said on Thursday it di...,"O,B-ORG,I-ORG,O,O,O,O,O,O,B-MISC,O,O,O,O,O,B-M..."
4,Germany 's representative to the European Unio...,"B-LOC,O,O,O,O,B-ORG,I-ORG,O,O,O,B-PER,I-PER,O,..."


Let's verify a sample sentence and its labels:

In [9]:
print("Sentence:", train_data.iloc[0].sentence)
print("Labels:", train_data.iloc[0].word_labels)

Sentence: EU rejects German call to boycott British lamb .
Labels: B-ORG,O,B-MISC,O,O,O,B-MISC,O,O


Now let's create the label-to-id mappings:

In [10]:
# Create label mappings
labels_to_ids = {label: idx for idx, label in enumerate(label_names)}
ids_to_labels = {idx: label for idx, label in enumerate(label_names)}
print("labels_to_ids:", labels_to_ids)
print("ids_to_labels:", ids_to_labels)

labels_to_ids: {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8}
ids_to_labels: {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC', 7: 'B-MISC', 8: 'I-MISC'}


In [11]:
# Use the train and test DataFrames we created earlier
data = train_data.copy()
print(f"Training data shape: {data.shape}")

Training data shape: (14041, 2)


In [12]:
data.head()

,sentence,word_labels
0,EU rejects German call to boycott British lamb .,"B-ORG,O,B-MISC,O,O,O,B-MISC,O,O"
1,Peter Blackburn,"B-PER,I-PER"
2,BRUSSELS 1996-08-22,"B-LOC,O"
3,The European Commission said on Thursday it di...,"O,B-ORG,I-ORG,O,O,O,O,O,O,B-MISC,O,O,O,O,O,B-M..."
4,Germany 's representative to the European Unio...,"B-LOC,O,O,O,O,B-ORG,I-ORG,O,O,O,B-PER,I-PER,O,..."


Let's look at the NER tags we'll use:

In [13]:
print(f"Number of labels: {len(labels_to_ids)}")
print("Labels:", list(labels_to_ids.keys()))

Number of labels: 9
Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


In [14]:
len(data)

14041

Let's verify that a random sentence and its corresponding tags are correct:

In [15]:
data.iloc[5].sentence

'" We do n\'t support any such recommendation because we do n\'t see any grounds for it , " the Commission \'s chief spokesman Nikolaus van der Pas told a news briefing .'

In [16]:
data.iloc[5].word_labels

'O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,B-ORG,O,O,O,B-PER,I-PER,I-PER,I-PER,O,O,O,O,O'

#### **Preparing the dataset and dataloader**

Now that our data is preprocessed, we can turn it into PyTorch tensors such that we can provide it to the model. Let's start by defining some key variables that will be used later on in the training/evaluation process:

In [17]:
MAX_LEN = 128
TRAIN_BATCH_SIZE = 4
VALID_BATCH_SIZE = 2
EPOCHS = 1
LEARNING_RATE = 1e-05
MAX_GRAD_NORM = 10
tokenizer = BertTokenizerFast.from_pretrained('prajjwal1/bert-tiny')

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

A tricky part of NER with BERT is that BERT relies on **wordpiece tokenization**, rather than word tokenization. This means that we should also define the labels at the wordpiece-level, rather than the word-level!

For example, if you have word like "Washington" which is labeled as "b-gpe", but it gets tokenized to "Wash", "##ing", "##ton", then one approach could be to handle this by only train the model on the tag labels for the first word piece token of a word (i.e. only label "Wash" with "b-gpe"). This is what was done in the original BERT paper, see Github discussion [here](https://github.com/huggingface/transformers/issues/64#issuecomment-443703063).

Note that this is a **design decision**. You could also decide to propagate the original label of the word to all of its word pieces and let the model train on this. In that case, the model should be able to produce the correct labels for each individual wordpiece. This was done in [this NER tutorial with BERT](https://github.com/chambliss/Multilingual_NER/blob/master/python/utils/main_utils.py#L118). Another design decision could be to give the first wordpiece of each word the original word label, and then use the label “X” for all subsequent subwords of that word. All of them seem to lead to good performance.

Below, we define a regular PyTorch [dataset class](https://pytorch.org/docs/stable/data.html) (which transforms examples of a dataframe to PyTorch tensors). Here, each sentence gets tokenized, the special tokens that BERT expects are added, the tokens are padded or truncated based on the max length of the model, the attention mask is created and the labels are created based on the dictionary which we defined above. Word pieces that should be ignored have a label of -100 (which is the default `ignore_index` of PyTorch's [CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)).

For more information about BERT's inputs, see [here](https://huggingface.co/transformers/glossary.html).








In [18]:
class DataSequence(Dataset):
  __getitems__ = None
  def __init__(self, dataframe, tokenizer, max_len):
        self.len = len(dataframe)
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

  def __getitem__(self, index):
        # step 1: get the sentence and word labels
        sentence = self.dataframe.iloc[index].sentence.strip().split()
        word_labels = self.dataframe.iloc[index].word_labels.split(",")

        # step 2: use tokenizer to encode sentence (includes padding/truncation up to max length)
        encoding = self.tokenizer(sentence,
                             is_split_into_words=True,
                             return_offsets_mapping=True,
                             padding='max_length',
                             truncation=True,
                             max_length=self.max_len)

        # step 3: create token labels only for first word pieces of each tokenized word
        labels = [labels_to_ids[label] for label in word_labels]
        # create an empty array of -100 of length max_length
        encoded_labels = np.ones(len(encoding["offset_mapping"]), dtype=int) * -100

        # set only labels whose first offset position is 0 and the second is not 0
        i = 0
        for idx, mapping in enumerate(encoding["offset_mapping"]):
          if mapping[0] == 0 and mapping[1] != 0:
            # overwrite label
            encoded_labels[idx] = labels[i]
            i += 1

        # step 4: turn everything into PyTorch tensors
        item = {key: torch.as_tensor(val) for key, val in encoding.items()}
        item['labels'] = torch.as_tensor(encoded_labels)

        return item

  def __len__(self):
        return self.len

Now, based on the class we defined above, we can create 2 datasets, one for training and one for testing. We use the dataset's predefined train and test splits:

In [19]:
# We already have train and test splits from the HuggingFace dataset
train_dataset = train_data.reset_index(drop=True)
test_dataset = test_data.reset_index(drop=True)

print("TRAIN Dataset: {}".format(train_dataset.shape))
print("TEST Dataset: {}".format(test_dataset.shape))

training_set = DataSequence(train_dataset, tokenizer, MAX_LEN)
testing_set = DataSequence(test_dataset, tokenizer, MAX_LEN)

TRAIN Dataset: (14041, 2)
TEST Dataset: (3453, 2)


Let's have a look at the first training example:

In [20]:
training_set[0]

{'input_ids': tensor([  101,  7327, 19164,  2446,  2655,  2000, 17757,  2329, 12559,  1012,
           102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,   

Let's verify that the input ids and corresponding targets are correct:

In [21]:
for token, label in zip(tokenizer.convert_ids_to_tokens(training_set[0]["input_ids"]), training_set[0]["labels"]):
  print('{0:10}  {1}'.format(token, label))

[CLS]       -100
eu          3
rejects     0
german      7
call        0
to          0
boycott     0
british     7
lamb        0
.           0
[SEP]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]  

Now, let's define the corresponding PyTorch dataloaders:

In [22]:
train_params = {'batch_size': TRAIN_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

test_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

training_loader = DataLoader(training_set, **train_params)
testing_loader = DataLoader(testing_set, **test_params)

#### **Defining the model**

Here we define the model, BertForTokenClassification, and load it with the pretrained weights of 'prajjwal1/bert-tiny'. The only thing we need to additionally specify is the number of labels (as this will determine the architecture of the classification head).

Note that only the base layers are initialized with the pretrained weights. The token classification head of top has just randomly initialized weights, which we will train, together with the pretrained weights, using our labelled dataset. This is also printed as a warning when you run the code cell below.

Then, we move the model to the GPU.

In [23]:
model = BertForTokenClassification.from_pretrained('prajjwal1/bert-tiny', num_labels=len(labels_to_ids))
model.to(device)

config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/37 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignore

model.safetensors:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-1): 2 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=128, out_features=128, bias=True)
              (key): Linear(in_features=128, out_features=128, bias=True)
              (value): Linear(in_features=128, out_features=128, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=128, out_features=128, bias=True)
              (LayerNorm): LayerNorm((128,), eps=1e-12, elem

#### **Training the model**

Before training the model, let's perform a sanity check, which I learned thanks to Andrej Karpathy's wonderful [cs231n course](http://cs231n.stanford.edu/) at Stanford (see also his [blog post about debugging neural networks](http://karpathy.github.io/2019/04/25/recipe/)).

Why? Because we are using cross entropy loss. The cross entropy loss is defined as -ln(probability score of the model for the correct class). In the beginning, the weights are random, so the probability distribution for all of the classes for a given token will be uniform, meaning that the probability for the correct class will be near 1/7. The loss for a given token will thus be -ln(1/7).

Let's verify this:

In [24]:
inputs = training_set[2]
input_ids = inputs["input_ids"].unsqueeze(0)
attention_mask = inputs["attention_mask"].unsqueeze(0)
labels = inputs["labels"].unsqueeze(0)

input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)
labels = labels.to(device)

outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
initial_loss = outputs[0]
initial_loss

tensor(2.3232, device='cuda:0', grad_fn=<NllLossBackward0>)

This looks good. Let's also verify that the logits of the neural network have a shape of (batch_size, sequence_length, num_labels):

In [25]:
tr_logits = outputs[1]
tr_logits.shape

torch.Size([1, 128, 9])

Next, we define the optimizer. Here, we are just going to use Adam with a default learning rate. One can also decide to use more advanced ones such as AdamW (Adam with weight decay fix), which is [included](https://huggingface.co/transformers/main_classes/optimizer_schedules.html) in the Transformers repository, and a learning rate scheduler, but we are not going to do that here.

In [26]:
optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)

Now let's define a regular PyTorch training function. It is partly based on [a really good repository about multilingual NER](https://github.com/chambliss/Multilingual_NER/blob/master/python/utils/main_utils.py#L344).

In [27]:
# Defining the training function on the 80% of the dataset for tuning the bert model
def train(epoch):
    tr_loss, tr_accuracy = 0, 0
    nb_tr_examples, nb_tr_steps = 0, 0
    tr_preds, tr_labels = [], []
    # put model in training mode
    model.train()

    for idx, batch in enumerate(training_loader):

        ids = batch['input_ids'].to(device, dtype = torch.long)
        mask = batch['attention_mask'].to(device, dtype = torch.long)
        labels = batch['labels'].to(device, dtype = torch.long)

        outputs = model(input_ids=ids, attention_mask=mask, labels=labels)
        loss, tr_logits = outputs.loss, outputs.logits
        tr_loss += loss.item()

        nb_tr_steps += 1
        nb_tr_examples += labels.size(0)

        if idx % 100==0:
            loss_step = tr_loss/nb_tr_steps
            print(f"Training loss per 100 training steps: {loss_step}")

        # compute training accuracy
        flattened_targets = labels.view(-1) # shape (batch_size * seq_len,)
        active_logits = tr_logits.view(-1, model.num_labels) # shape (batch_size * seq_len, num_labels)
        flattened_predictions = torch.argmax(active_logits, axis=1) # shape (batch_size * seq_len,)

        # only compute accuracy at active labels
        active_accuracy = labels.view(-1) != -100 # shape (batch_size, seq_len)
        #active_labels = torch.where(active_accuracy, labels.view(-1), torch.tensor(-100).type_as(labels))

        labels = torch.masked_select(flattened_targets, active_accuracy)
        predictions = torch.masked_select(flattened_predictions, active_accuracy)

        tr_labels.extend(labels)
        tr_preds.extend(predictions)

        tmp_tr_accuracy = accuracy_score(labels.cpu().numpy(), predictions.cpu().numpy())
        tr_accuracy += tmp_tr_accuracy

        # gradient clipping
        torch.nn.utils.clip_grad_norm_(
            parameters=model.parameters(), max_norm=MAX_GRAD_NORM
        )

        # backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    epoch_loss = tr_loss / nb_tr_steps
    tr_accuracy = tr_accuracy / nb_tr_steps
    print(f"Training loss epoch: {epoch_loss}")
    print(f"Training accuracy epoch: {tr_accuracy}")

And let's train the model!

In [28]:
for epoch in range(EPOCHS):
    print(f"Training epoch: {epoch + 1}")
    train(epoch)

Training epoch: 1
Training loss per 100 training steps: 2.1505472660064697
Training loss per 100 training steps: 1.7553141034475648
Training loss per 100 training steps: 1.485662357131047
Training loss per 100 training steps: 1.3172208862843306
Training loss per 100 training steps: 1.2146908260343081
Training loss per 100 training steps: 1.1472203418642224
Training loss per 100 training steps: 1.0896991760084118
Training loss per 100 training steps: 1.0497907616527546
Training loss per 100 training steps: 1.0159126538387995
Training loss per 100 training steps: 0.9805781966639676
Training loss per 100 training steps: 0.9514865147007572
Training loss per 100 training steps: 0.9257497733609444
Training loss per 100 training steps: 0.9019802539399224
Training loss per 100 training steps: 0.8805590984365007
Training loss per 100 training steps: 0.8567555160989598
Training loss per 100 training steps: 0.8366953503442239
Training loss per 100 training steps: 0.8167898096027857
Training loss 

#### **Evaluating the model**

Now that we've trained our model, we can evaluate its performance on the held-out test set. Note that here, no gradient updates are performed, the model just outputs its logits.

In [29]:
def valid(model, testing_loader):
    # put model in evaluation mode
    model.eval()

    eval_loss, eval_accuracy = 0, 0
    nb_eval_examples, nb_eval_steps = 0, 0
    eval_preds, eval_labels = [], []

    with torch.no_grad():
        for idx, batch in enumerate(testing_loader):

            ids = batch['input_ids'].to(device, dtype = torch.long)
            mask = batch['attention_mask'].to(device, dtype = torch.long)
            labels = batch['labels'].to(device, dtype = torch.long)

            outputs = model(input_ids=ids, attention_mask=mask, labels=labels)
            loss, eval_logits = outputs.loss, outputs.logits

            eval_loss += loss.item()

            nb_eval_steps += 1
            nb_eval_examples += labels.size(0)

            if idx % 100==0:
                loss_step = eval_loss/nb_eval_steps
                print(f"Validation loss per 100 evaluation steps: {loss_step}")

            # compute evaluation accuracy
            flattened_targets = labels.view(-1) # shape (batch_size * seq_len,)
            active_logits = eval_logits.view(-1, model.num_labels) # shape (batch_size * seq_len, num_labels)
            flattened_predictions = torch.argmax(active_logits, axis=1) # shape (batch_size * seq_len,)

            # only compute accuracy at active labels
            active_accuracy = labels.view(-1) != -100 # shape (batch_size, seq_len)

            labels = torch.masked_select(flattened_targets, active_accuracy)
            predictions = torch.masked_select(flattened_predictions, active_accuracy)

            eval_labels.extend(labels)
            eval_preds.extend(predictions)

            tmp_eval_accuracy = accuracy_score(labels.cpu().numpy(), predictions.cpu().numpy())
            eval_accuracy += tmp_eval_accuracy

    labels = [ids_to_labels[id.item()] for id in eval_labels]
    predictions = [ids_to_labels[id.item()] for id in eval_preds]

    eval_loss = eval_loss / nb_eval_steps
    eval_accuracy = eval_accuracy / nb_eval_steps
    print(f"Validation Loss: {eval_loss}")
    print(f"Validation Accuracy: {eval_accuracy}")

    return labels, predictions

As we can see below, token-level accuracy looks high, but as the next cell shows, the per-entity F1 scores are more modest, especially since we're using a small model (bert-tiny) trained for just one epoch.

In [30]:
labels, predictions = valid(model, testing_loader)

Validation loss per 100 evaluation steps: 0.47349977493286133
Validation loss per 100 evaluation steps: 0.41999641704456053
Validation loss per 100 evaluation steps: 0.4049537093726112
Validation loss per 100 evaluation steps: 0.39104503179680666
Validation loss per 100 evaluation steps: 0.39568676940743763
Validation loss per 100 evaluation steps: 0.39202165789530424
Validation loss per 100 evaluation steps: 0.3848650746400811
Validation loss per 100 evaluation steps: 0.3833971607861522
Validation loss per 100 evaluation steps: 0.38203098654709805
Validation loss per 100 evaluation steps: 0.38201354217863376
Validation loss per 100 evaluation steps: 0.3854264399682457
Validation loss per 100 evaluation steps: 0.38720999028936337
Validation loss per 100 evaluation steps: 0.3876000830560898
Validation loss per 100 evaluation steps: 0.3932396965609616
Validation loss per 100 evaluation steps: 0.3941818271257744
Validation loss per 100 evaluation steps: 0.3948430657079028
Validation loss 

In [31]:
from seqeval.metrics import classification_report

# labels/predictions already come back as flat lists of string tags.
# seqeval wants list-of-lists, so just wrap each in an outer list.
print(classification_report([labels], [predictions]))

              precision    recall  f1-score   support

         LOC       0.57      0.71      0.63      1661
        MISC       0.67      0.29      0.41       702
         ORG       0.37      0.24      0.29      1661
         PER       0.34      0.51      0.41      1611

   micro avg       0.44      0.46      0.45      5635
   macro avg       0.49      0.44      0.43      5635
weighted avg       0.46      0.46      0.44      5635



#### **Inference**

The fun part is when we can quickly test the model on new, unseen sentences.
Here, we use the prediction of the **first word piece of every word** (which is how the model was trained).

*In other words, the code below does not take into account when predictions of different word pieces that belong to the same word do not match.*

In [32]:
sentence = "@HuggingFace is a company based in New York, but is also has employees working in Paris"

inputs = tokenizer(sentence.split(),
                    is_split_into_words=True,
                    return_offsets_mapping=True,
                    padding='max_length',
                    truncation=True,
                    max_length=MAX_LEN,
                    return_tensors="pt")

# move to gpu
ids = inputs["input_ids"].to(device)
mask = inputs["attention_mask"].to(device)
# forward pass
outputs = model(ids, attention_mask=mask)
logits = outputs.logits

active_logits = logits.view(-1, model.num_labels) # shape (batch_size * seq_len, num_labels)
flattened_predictions = torch.argmax(active_logits, axis=1) # shape (batch_size*seq_len,) - predictions at the token level

tokens = tokenizer.convert_ids_to_tokens(ids.squeeze().tolist())
token_predictions = [ids_to_labels[i] for i in flattened_predictions.cpu().numpy()]
wp_preds = list(zip(tokens, token_predictions)) # list of tuples. Each tuple = (wordpiece, prediction)

prediction = []
for token_pred, mapping in zip(wp_preds, inputs["offset_mapping"].squeeze().tolist()):
  #only predictions on first word pieces are important
  if mapping[0] == 0 and mapping[1] != 0:
    prediction.append(token_pred[1])
  else:
    continue

print(sentence.split())
print(prediction)

['@HuggingFace', 'is', 'a', 'company', 'based', 'in', 'New', 'York,', 'but', 'is', 'also', 'has', 'employees', 'working', 'in', 'Paris']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-LOC']


#### **Saving the model for future use**

Finally, let's save the vocabulary (.txt) file, model weights (.bin) and the model's configuration (.json) to a directory, so that both the tokenizer and model can be re-loaded using the `from_pretrained()` class method.


In [33]:
import os

directory = "./model"

if not os.path.exists(directory):
    os.makedirs(directory)

# save vocabulary of the tokenizer
tokenizer.save_vocabulary(directory)
# save the model weights and its configuration file
model.save_pretrained(directory)
print('All files saved')
print('This tutorial is completed')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

All files saved
This tutorial is completed
